# 07 - Preparação para deep learning e redes neurais (módulos 10 e 11)

Janelamento temporal configurável (módulo 10) e comparação de SimpleRNN/LSTM/GRU/BiLSTM/CNN+LSTM/Transformer (módulo 11), com um exemplo de ajuste automático de hiperparâmetros.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from harmonic_signatures.config import PROCESSED_DIR
from harmonic_signatures.deep_learning.windowing import make_sequence_windows
from harmonic_signatures.deep_learning.architectures import ARCHITECTURES
from harmonic_signatures.deep_learning.training import train_classifier
from harmonic_signatures.deep_learning.tuning import random_search
from harmonic_signatures.deep_learning.architectures import build_lstm

harm_matrix = pd.read_parquet(PROCESSED_DIR / 'combined_matrix.parquet')
feature_cols = [c for c in harm_matrix.columns if c.startswith('H') and c.endswith('_avg') and '_harm' not in c and '_inter' not in c][:20]

## Janelas de sequência (window_size=8, stride=4)

In [2]:
wd = make_sequence_windows(harm_matrix, feature_cols, window_size=8, stride=4)
print('janelas:', wd.X.shape)

idx = np.arange(len(wd.X))
train_idx, val_idx = train_test_split(idx, test_size=0.25, random_state=0, stratify=wd.groups)
X_train, X_val = wd.X[train_idx], wd.X[val_idx]
y_train, y_val = wd.y_class[train_idx], wd.y_class[val_idx]

mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
std = X_train.reshape(-1, X_train.shape[-1]).std(axis=0); std[std == 0] = 1.0
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

janelas: (4853, 8, 20)


## Comparação das 6 arquiteturas (poucas épocas — ajustar para produção)

In [3]:
rows = []
for name, builder in ARCHITECTURES.items():
    result = train_classifier(builder, {}, X_train, y_train, X_val, y_val, epochs=6, batch_size=64, verbose=0)
    rows.append({'arquitetura': name, 'val_accuracy': result.val_accuracy, 'val_loss': result.val_loss})
pd.DataFrame(rows).sort_values('val_accuracy', ascending=False)

,arquitetura,val_accuracy,val_loss
5,transformer,0.844316,0.358957
3,bilstm,0.817957,0.429641
4,cnn_lstm,0.810544,0.437926
1,lstm,0.803130,0.503853
0,simple_rnn,0.788303,0.569693
2,gru,0.780066,0.544051


## Ajuste automático de hiperparâmetros (random search) — LSTM

In [4]:
grid = {'units': [32, 64, 128], 'dropout': [0.1, 0.2, 0.3]}
trials_df, best_params, best_model = random_search(build_lstm, grid, X_train, y_train, X_val, y_val, n_trials=4, epochs=4)
print('melhores parâmetros:', best_params)
trials_df

melhores parâmetros: {'units': 128, 'dropout': 0.1}


,trial,units,dropout,val_accuracy,val_loss
0,0,128,0.1,0.812191,0.445173
1,2,64,0.1,0.807249,0.504710
2,3,32,0.1,0.766063,0.811327
3,1,32,0.3,0.710873,0.887464
